In [21]:
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, GlobalAveragePooling1D
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [22]:
df = pd.read_csv("dice_com-job_us_sample.csv")

df.head()

,advertiserurl,company,employmenttype_jobstatus,jobdescription,jobid,joblocation_address,jobtitle,postdate
0,https://www.dice.com/jobs/detail/AUTOMATION-TE...,"Digital Intelligence Systems, LLC","C2H Corp-To-Corp, C2H Independent, C2H W2, 3 M...",Looking for Selenium engineers...must have sol...,Dice Id : 10110693,"Atlanta, GA",AUTOMATION TEST ENGINEER,1 hour ago
1,https://www.dice.com/jobs/detail/Information-S...,University of Chicago/IT Services,Full Time,The University of Chicago has a rapidly growin...,Dice Id : 10114469,"Chicago, IL",Information Security Engineer,1 week ago
2,https://www.dice.com/jobs/detail/Business-Solu...,"Galaxy Systems, Inc.",Full Time,"GalaxE.SolutionsEvery day, our solutions affec...",Dice Id : CXGALXYS,"Schaumburg, IL",Business Solutions Architect,2 weeks ago
3,https://www.dice.com/jobs/detail/Java-Develope...,TransTech LLC,Full Time,Java DeveloperFull-time/direct-hireBolingbrook...,Dice Id : 10113627,"Bolingbrook, IL","Java Developer (mid level)- FT- GREAT culture,...",2 weeks ago
4,https://www.dice.com/jobs/detail/DevOps-Engine...,Matrix Resources,Full Time,Midtown based high tech firm has an immediate ...,Dice Id : matrixga,"Atlanta, GA",DevOps Engineer,48 minutes ago


In [23]:
print(df.columns)

Index(['advertiserurl', 'company', 'employmenttype_jobstatus',
       'jobdescription', 'jobid', 'joblocation_address', 'jobtitle',
       'postdate'],
      dtype='str')


In [24]:
df = df[['jobtitle', 'jobdescription']]
df.head()

,jobtitle,jobdescription
0,AUTOMATION TEST ENGINEER,Looking for Selenium engineers...must have sol...
1,Information Security Engineer,The University of Chicago has a rapidly growin...
2,Business Solutions Architect,"GalaxE.SolutionsEvery day, our solutions affec..."
3,"Java Developer (mid level)- FT- GREAT culture,...",Java DeveloperFull-time/direct-hireBolingbrook...
4,DevOps Engineer,Midtown based high tech firm has an immediate ...


In [25]:
df = df[['jobtitle','jobdescription']]

In [26]:
df = df.dropna()

print(df.shape)

(9000, 2)


In [27]:


top_jobs = df['jobtitle'].value_counts().head(20).index

df = df[df['jobtitle'].isin(top_jobs)]

print("Dataset Shape:", df.shape)

print("\nTop 20 Job Titles:")

print(df['jobtitle'].value_counts())

Dataset Shape: (581, 2)

Top 20 Job Titles:
jobtitle
Java Developer                   75
Network Engineer                 54
Project Manager                  53
Business Analyst                 45
Software Engineer                45
Software Development Engineer    28
Systems Engineer                 26
.Net Developer                   25
Web Developer                    24
Systems Administrator            24
Senior Java Developer            22
Data Analyst                     21
Software Developer               19
DevOps Engineer                  18
Front End Developer              18
Data Scientist                   18
Android Developer                18
.NET Developer                   18
Business Systems Analyst         15
Senior Network Engineer          15
Name: count, dtype: int64


In [28]:
X = df['jobdescription']
y = df['jobtitle']

In [29]:
label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("Total Classes:", len(label_encoder.classes_))

Total Classes: 20


In [30]:
tokenizer = Tokenizer(num_words=10000)

tokenizer.fit_on_texts(X)

sequences = tokenizer.texts_to_sequences(X)

In [31]:
max_len = 200

X_pad = pad_sequences(
    sequences,
    maxlen=max_len,
    padding='post'
)

print(X_pad.shape)

(581, 200)


In [32]:
X_train, X_test, y_train, y_test = train_test_split(
    X_pad,
    y_encoded,
    test_size=0.2,
    random_state=42
)

In [33]:
model = Sequential([
    Embedding(10000, 64, input_length=max_len),
    GlobalAveragePooling1D(),
    Dense(128, activation='relu'),
    Dense(len(label_encoder.classes_), activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

c:\Users\navee\Desktop\Data_Scientist\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [34]:
history = model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=1
)

Epoch 1/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 3s 47ms/step - accuracy: 0.1034 - loss: 2.9868 - val_accuracy: 0.0940 - val_loss: 2.9589
Epoch 2/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.1034 - loss: 2.9445 - val_accuracy: 0.1111 - val_loss: 2.9007
Epoch 3/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.1228 - loss: 2.8877 - val_accuracy: 0.1368 - val_loss: 2.8283
Epoch 4/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.1250 - loss: 2.8452 - val_accuracy: 0.1453 - val_loss: 2.7918
Epoch 5/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.1272 - loss: 2.8004 - val_accuracy: 0.1368 - val_loss: 2.7701
Epoch 6/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.1659 - loss: 2.7546 - val_accuracy: 0.1795 - val_loss: 2.7433
Epoch 7/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.1875 - loss: 2.7049 - val_accuracy: 0.1624 - val_loss: 2.7100
Epoch 8/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.2392 - loss: 2.6369 - val_accuracy: 0.2479 - v

In [35]:
loss, acc = model.evaluate(X_test, y_test)

print("Test Accuracy:", acc * 100)

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.4444 - loss: 1.8510
Test Accuracy: 44.44444477558136


In [36]:
model.save("job_model.h5") 

print("Model Saved Successfully")

Model Saved Successfully


In [37]:
with open("tokenizer.pkl","wb") as f:
    pickle.dump(tokenizer,f)

print("Tokenizer Saved")

Tokenizer Saved


In [38]:
with open("label_encoder.pkl","wb") as f:
    pickle.dump(label_encoder,f)

print("Label Encoder Saved")

Label Encoder Saved


In [39]:
sample = """
Python developer with machine learning,
SQL, Pandas, NumPy and Deep Learning skills
"""

seq = tokenizer.texts_to_sequences([sample])

pad = pad_sequences(seq,maxlen=max_len,padding='post')

pred = model.predict(pad)

job = label_encoder.inverse_transform([np.argmax(pred)])

print("Recommended Job:", job[0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step
Recommended Job: Project Manager
